In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn shap lime xgboost

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')
warnings.filterwarnings('ignore', message='.*lbfgs.*', module='sklearn')

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, precision_score, recall_score, f1_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import shap
import lime
import lime.lime_tabular
import joblib
import os

In [ ]:
# Configure data path - change this single variable to point to your dataset
DATA_DIR = os.getcwd()
DATA_FILE = 'loan_approval_dataset.csv'
DATA_PATH = os.path.join(DATA_DIR, DATA_FILE)

# Fallback: search common locations
if not os.path.exists(DATA_PATH):
    search_paths = [
        os.path.join(os.path.expanduser('~'), 'Downloads', DATA_FILE),
        os.path.join(os.path.expanduser('~'), 'Downloads', 'Loan Approval Prediction', DATA_FILE),
    ]
    for p in search_paths:
        if os.path.exists(p):
            DATA_PATH = p
            break

print(f"Loading data from: {DATA_PATH}")
data = pd.read_csv(DATA_PATH)
data.columns = data.columns.str.strip()
print(f"Dataset loaded: {data.shape[0]} rows, {data.shape[1]} columns")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# ============================================
# CLASS BALANCE CHECK
# ============================================
print("=== Target Variable Distribution ===")
print(data['loan_status'].value_counts())
print()
print(data['loan_status'].value_counts(normalize=True).round(4) * 100)

plt.figure(figsize=(6, 4))
data['loan_status'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Loan Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

majority_pct = data['loan_status'].value_counts(normalize=True).max()
if majority_pct > 0.7:
    print(f"WARNING: Dataset is imbalanced ({majority_pct:.1%} majority class)")
    print("Consider using stratified splitting, SMOTE, or class_weight='balanced'")
else:
    print("Class distribution is reasonably balanced")

In [ ]:
# ============================================
# DATA QUALITY: NEGATIVE VALUES
# ============================================
print("=== Checking for Negative Values in Asset Columns ===")
asset_cols = ['residential_assets_value', 'commercial_assets_value',
              'luxury_assets_value', 'bank_asset_value']

for col in asset_cols:
    neg_count = (data[col] < 0).sum()
    if neg_count > 0:
        print(f"WARNING: {col}: {neg_count} negative values (min={data[col].min():,.0f})")
    else:
        print(f"OK: {col}: no negative values")

# ============================================
# OUTLIER ANALYSIS (IQR Method)
# ============================================
print("\n=== Outlier Detection (IQR Method) ===")
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'loan_id']

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((data[col] < lower) | (data[col] > upper)).sum()
    pct = n_outliers / len(data) * 100
    if n_outliers > 0:
        print(f"  {col}: {n_outliers} outliers ({pct:.1f}%)")

# Boxplots
n_cols_plot = min(4, len(numeric_cols))
n_rows_plot = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(16, 3 * n_rows_plot))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for i, col in enumerate(numeric_cols):
    if i < len(axes_flat):
        data.boxplot(column=col, ax=axes_flat[i])
        axes_flat[i].set_title(col, fontsize=8)
for j in range(len(numeric_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle('Boxplot Outlier Analysis', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================
# DUPLICATE ROW DETECTION
# ============================================
n_dupes = data.duplicated().sum()
if n_dupes > 0:
    print(f"\nWARNING: {n_dupes} duplicate rows detected ({n_dupes/len(data)*100:.1f}%)")
    print("Consider: data = data.drop_duplicates()")
else:
    print(f"\nOK: No duplicate rows detected")

In [ ]:
# 1. Correlation Heatmap
plt.figure(figsize=(12, 8))
numeric_data = data.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_data.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

# 2. Distribution of Loan Status
plt.figure(figsize=(6, 4))
sns.countplot(x='loan_status', data=data, palette='Set2')
plt.title('Distribution of Loan Status')
plt.show()

In [ ]:
# ============================================
# FEATURE DISTRIBUTION BY CLASS
# ============================================
plot_features = [
    col for col in data.select_dtypes(include=[np.number]).columns
    if col not in ['loan_id', 'loan_status']
]

class_labels = sorted(data['loan_status'].unique(), key=str)
class_display = []
for lbl in class_labels:
    lbl_clean = str(lbl).strip().lower()
    if lbl_clean in ('rejected', '0'):
        class_display.append((lbl, '#e74c3c', 'Rejected'))
    else:
        class_display.append((lbl, '#2ecc71', 'Approved'))

n_feats = len(plot_features)
n_cols = 3
n_rows = (n_feats + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(plot_features):
    ax = axes[i]
    for label, color, name in class_display:
        subset = data[data['loan_status'] == label][col]
        ax.hist(subset, bins=30, alpha=0.5, label=name, color=color, density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distribution by Loan Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
data.isnull().sum()

In [ ]:
# Verify data shape before preprocessing
print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head(3)

In [ ]:
# Column names already stripped at load time (cell 3)
print(f"Columns: {list(data.columns)}")

In [ ]:
# Feature engineering
data['income_loan_ratio'] = np.where(
    data['loan_amount'] == 0, 0,
    data['income_annum'] / data['loan_amount']
)
data['total_assets'] = (data['residential_assets_value'] +
                         data['commercial_assets_value'] +
                         data['luxury_assets_value'] +
                         data['bank_asset_value'])

In [ ]:
# ============================================
# LABEL ENCODING - SINGLE SOURCE OF TRUTH
# ============================================
# Encoding scheme (documented once, used everywhere):
#   education:     Graduate = 1, Not Graduate = 0
#   self_employed: Yes = 1, No = 0
#   loan_status:   Approved = 1, Rejected = 0
# ============================================

if data['education'].dtype == 'object':
    data['education'] = data['education'].str.strip()
    data['education'] = data['education'].map({'Graduate': 1, 'Not Graduate': 0})

if data['self_employed'].dtype == 'object':
    data['self_employed'] = data['self_employed'].str.strip()
    data['self_employed'] = data['self_employed'].map({'Yes': 1, 'No': 0})

if data['loan_status'].dtype == 'object':
    data['loan_status'] = data['loan_status'].str.strip()
    data['loan_status'] = data['loan_status'].map({'Approved': 1, 'Rejected': 0})

assert data[['education', 'self_employed', 'loan_status']].isnull().sum().sum() == 0, \
    "ERROR: Mapping introduced NaN values"

print("Label encoding complete:")
print(f"  education unique:     {sorted(data['education'].unique())}")
print(f"  self_employed unique: {sorted(data['self_employed'].unique())}")
print(f"  loan_status unique:   {sorted(data['loan_status'].unique())}")

In [ ]:
# Drop loan_id if present (errors='ignore' handles missing/already-dropped column)
data = data.drop("loan_id", axis=1, errors='ignore')

In [ ]:
data.head()  # loan id is not there

In [ ]:
# ============================================
# FEATURE / TARGET SPLIT
# ============================================
X = data.drop('loan_status', axis=1)
y = data['loan_status']

print(f"Features: {list(X.columns)}")
print(f"Feature count: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts()}")

# ============================================
# STRATIFIED TRAIN/TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# ============================================
# STANDARD SCALING (APPLIED TO ALL MODELS)
# ============================================
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)
print("Scaling applied to train and test sets")

# ============================================
# CROSS-VALIDATION SETUP
# ============================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("5-fold stratified cross-validation configured")

# DECISION TREE

In [ ]:
# ============================================
# DECISION TREE — HYPERPARAMETER TUNING
# ============================================
# Note: Tree models are scale-invariant. Scaling has no effect on
# their predictions but is applied uniformly for pipeline simplicity.
dt_param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=cv,
    scoring='accuracy',  # Consider 'f1' or 'roc_auc' for imbalanced data
    n_jobs=-1,
    verbose=0
)
dt_grid.fit(X_train_scaled, y_train)
dt_model = dt_grid.best_estimator_  # Refit on full X_train_scaled (refit=True default)

dt_pred = dt_model.predict(X_test_scaled)
dt_prob = dt_model.predict_proba(X_test_scaled)[:, 1]

# Use GridSearchCV's internal CV results (no redundant cross_val_score)
dt_cv_mean = dt_grid.best_score_
dt_cv_std = dt_grid.cv_results_['std_test_score'][dt_grid.best_index_]

print("=== Decision Tree Results ===")
print(f"Best Params:   {dt_grid.best_params_}")
print(f"CV Accuracy:   {dt_cv_mean:.4f} +/- {dt_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, dt_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, dt_pred, target_names=['Rejected', 'Approved']))

In [ ]:
dt_cm = confusion_matrix(y_test, dt_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=dt_cm)
disp.plot(cmap='Blues')
plt.title("Decision Tree Confusion Matrix")
plt.show()

In [ ]:
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_prob)
dt_roc_auc = auc(dt_fpr, dt_tpr)

plt.figure(figsize=(7, 5))
plt.plot(dt_fpr, dt_tpr, label=f"AUC = {dt_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree')
plt.legend()
plt.show()

In [ ]:
dt_importance = dt_model.feature_importances_

feat_imp = pd.Series(dt_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance - Decision Tree")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# LOGISTIC REGRESSION

In [ ]:
# ============================================
# LOGISTIC REGRESSION — HYPERPARAMETER TUNING
# ============================================
# Use a list of dicts to handle solver-penalty compatibility:
#   - 'lbfgs' supports only 'l2'
#   - 'liblinear' supports both 'l1' and 'l2'
lr_param_grid = [
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']},
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1'], 'solver': ['liblinear']},
]

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
lr_grid.fit(X_train_scaled, y_train)
lr_model = lr_grid.best_estimator_

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_cv_mean = lr_grid.best_score_
lr_cv_std = lr_grid.cv_results_['std_test_score'][lr_grid.best_index_]

print("=== Logistic Regression Results ===")
print(f"Best Params:   {lr_grid.best_params_}")
print(f"CV Accuracy:   {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, lr_pred, target_names=['Rejected', 'Approved']))

In [ ]:
lr_cm = confusion_matrix(y_test, lr_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=lr_cm)
disp.plot(cmap='Greens')
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
lr_roc_auc = auc(lr_fpr, lr_tpr)

plt.figure(figsize=(7, 5))
plt.plot(lr_fpr, lr_tpr, label=f"AUC = {lr_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.show()

In [ ]:
lr_importance = lr_model.coef_[0]

feat_imp = pd.Series(lr_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance (Coefficients) - Logistic Regression")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

# RANDOM FOREST

In [ ]:
# ============================================
# RANDOM FOREST — HYPERPARAMETER TUNING
# ============================================
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train_scaled, y_train)
rf_model = rf_grid.best_estimator_

rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_cv_mean = rf_grid.best_score_
rf_cv_std = rf_grid.cv_results_['std_test_score'][rf_grid.best_index_]

print("=== Random Forest Results ===")
print(f"Best Params:   {rf_grid.best_params_}")
print(f"CV Accuracy:   {rf_cv_mean:.4f} +/- {rf_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, rf_pred, target_names=['Rejected', 'Approved']))

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=rf_cm)
disp.plot(cmap='Oranges')
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)
rf_roc_auc = auc(rf_fpr, rf_tpr)

plt.figure(figsize=(7, 5))
plt.plot(rf_fpr, rf_tpr, label=f"AUC = {rf_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.show()

In [ ]:
rf_importance = rf_model.feature_importances_

feat_imp = pd.Series(rf_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# XGBOOST

In [ ]:
# ============================================
# XGBOOST — HYPERPARAMETER TUNING
# ============================================
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# n_jobs=1 on the estimator avoids thread contention with GridSearchCV's n_jobs=-1 (H3)
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=1),
    param_grid=xgb_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
xgb_grid.fit(X_train_scaled, y_train)
xgb_model = xgb_grid.best_estimator_

xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_cv_mean = xgb_grid.best_score_
xgb_cv_std = xgb_grid.cv_results_['std_test_score'][xgb_grid.best_index_]

print("=== XGBoost Results ===")
print(f"Best Params:   {xgb_grid.best_params_}")
print(f"CV Accuracy:   {xgb_cv_mean:.4f} +/- {xgb_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Rejected', 'Approved']))

# Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=xgb_cm)
disp.plot(cmap='Blues')
plt.title("XGBoost Confusion Matrix")
plt.show()

# ROC Curve
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_prob)
xgb_roc_auc = auc(xgb_fpr, xgb_tpr)
plt.figure(figsize=(7, 5))
plt.plot(xgb_fpr, xgb_tpr, label=f"AUC = {xgb_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost ROC Curve")
plt.legend()
plt.show()

# Feature Importance
xgb_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(xgb_importance_df['Feature'], xgb_importance_df['Importance'])
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()
print(xgb_importance_df)

# KNN

In [ ]:
# ============================================
# KNN — HYPERPARAMETER TUNING
# ============================================
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=knn_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
knn_grid.fit(X_train_scaled, y_train)
knn_model = knn_grid.best_estimator_

knn_pred = knn_model.predict(X_test_scaled)
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

knn_cv_mean = knn_grid.best_score_
knn_cv_std = knn_grid.cv_results_['std_test_score'][knn_grid.best_index_]

print("=== KNN Results ===")
print(f"Best Params:   {knn_grid.best_params_}")
print(f"CV Accuracy:   {knn_cv_mean:.4f} +/- {knn_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, knn_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, knn_pred, target_names=['Rejected', 'Approved']))

# Confusion Matrix
knn_cm = confusion_matrix(y_test, knn_pred)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=knn_cm)
disp.plot(cmap='Greens')
plt.title("KNN Confusion Matrix")
plt.show()

# ROC Curve
knn_fpr, knn_tpr, _ = roc_curve(y_test, knn_prob)
knn_roc_auc = auc(knn_fpr, knn_tpr)
plt.figure(figsize=(7, 5))
plt.plot(knn_fpr, knn_tpr, label=f"AUC = {knn_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN ROC Curve")
plt.legend()
plt.show()

# Permutation Importance
perm_importance = permutation_importance(
    knn_model, X_test_scaled, y_test, n_repeats=10, random_state=42
)
knn_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance.importances_mean
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(knn_importance_df['Feature'], knn_importance_df['Importance'])
plt.title("KNN Feature Importance (Permutation)")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()
print(knn_importance_df)

In [ ]:
# ============================================
# COMPREHENSIVE MODEL COMPARISON
# ============================================

model_info = {
    'Decision Tree':       {'pred': dt_pred,  'prob': dt_prob,  'cv_mean': dt_cv_mean,  'cv_std': dt_cv_std,  'model': dt_model},
    'Logistic Regression': {'pred': lr_pred,  'prob': lr_prob,  'cv_mean': lr_cv_mean,  'cv_std': lr_cv_std,  'model': lr_model},
    'Random Forest':       {'pred': rf_pred,  'prob': rf_prob,  'cv_mean': rf_cv_mean,  'cv_std': rf_cv_std,  'model': rf_model},
    'XGBoost':             {'pred': xgb_pred, 'prob': xgb_prob, 'cv_mean': xgb_cv_mean, 'cv_std': xgb_cv_std, 'model': xgb_model},
    'KNN':                 {'pred': knn_pred, 'prob': knn_prob, 'cv_mean': knn_cv_mean, 'cv_std': knn_cv_std, 'model': knn_model},
}

comparison_rows = []
for name, info in model_info.items():
    acc = accuracy_score(y_test, info['pred'])
    prec = precision_score(y_test, info['pred'], pos_label=1)   # Positive = Approved
    rec = recall_score(y_test, info['pred'], pos_label=1)
    f1 = f1_score(y_test, info['pred'], pos_label=1)
    fpr_vals, tpr_vals, _ = roc_curve(y_test, info['prob'])
    roc = auc(fpr_vals, tpr_vals)
    comparison_rows.append({
        'Model': name,
        'Test Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
        'AUC-ROC': round(roc, 4),
        'CV Accuracy (mean)': round(info['cv_mean'], 4),
        'CV Accuracy (std)': round(info['cv_std'], 4),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
print("Note: Precision/Recall/F1 computed for pos_label=1 (Approved)")
print(comparison_df.to_string())

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_df = comparison_df.sort_values('Test Accuracy')
axes[0].barh(sorted_df.index, sorted_df['Test Accuracy'], color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy Comparison')
for i, v in enumerate(sorted_df['Test Accuracy']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

axes[1].barh(
    comparison_df.index,
    comparison_df['CV Accuracy (mean)'],
    xerr=comparison_df['CV Accuracy (std)'],
    color='coral', capsize=5
)
axes[1].set_xlabel('CV Accuracy')
axes[1].set_title('Cross-Validation Accuracy (5-fold)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# MULTI-MODEL ROC CURVE OVERLAY
# ============================================
plt.figure(figsize=(10, 8))

model_roc_data = {
    'Decision Tree': (dt_prob, '#e74c3c'),
    'Logistic Regression': (lr_prob, '#3498db'),
    'Random Forest': (rf_prob, '#2ecc71'),
    'XGBoost': (xgb_prob, '#f39c12'),
    'KNN': (knn_prob, '#9b59b6'),
}

for name, (prob, color) in model_roc_data.items():
    fpr_vals, tpr_vals, _ = roc_curve(y_test, prob)
    roc_auc_val = auc(fpr_vals, tpr_vals)
    plt.plot(fpr_vals, tpr_vals, label=f"{name} (AUC = {roc_auc_val:.4f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# BEST MODEL SELECTION (using CV accuracy)
# ============================================
best_model_name = comparison_df['CV Accuracy (mean)'].idxmax()
best_model = model_info[best_model_name]['model']

print(f"Best Performing Model: {best_model_name}")
print(f"  Test Accuracy:  {comparison_df.loc[best_model_name, 'Test Accuracy']}")
print(f"  CV Accuracy:    {comparison_df.loc[best_model_name, 'CV Accuracy (mean)']} +/- {comparison_df.loc[best_model_name, 'CV Accuracy (std)']}")
print(f"  F1 Score:       {comparison_df.loc[best_model_name, 'F1 Score']}")
print(f"  AUC-ROC:        {comparison_df.loc[best_model_name, 'AUC-ROC']}")

In [ ]:
# ============================================
# SAVE BEST MODEL + SCALER
# ============================================
MODEL_SAVE_PATH = 'loan_model.pkl'
SCALER_SAVE_PATH = 'scaler.pkl'

joblib.dump(best_model, MODEL_SAVE_PATH)
joblib.dump(scaler, SCALER_SAVE_PATH)
joblib.dump(best_model_name, 'best_model_name.pkl')

print(f"Best model ({best_model_name}) saved to: {MODEL_SAVE_PATH}")
print(f"Scaler saved to: {SCALER_SAVE_PATH}")
print(f"Model name saved to: best_model_name.pkl")

In [ ]:
# ============================================
# SHAP EXPLAINABILITY - GLOBAL + LOCAL
# ============================================

def extract_shap_class1(sv, explainer):
    """Extract SHAP values for the positive class, handling all return formats."""
    if isinstance(sv, list):
        return sv[1], explainer.expected_value[1]
    sv_arr = np.array(sv)
    if sv_arr.ndim == 3:
        return sv_arr[:, :, 1], explainer.expected_value[1]
    ev = explainer.expected_value
    base = ev[1] if hasattr(ev, '__len__') and len(ev) > 1 else float(ev)
    return sv_arr, base

if best_model_name in ['Decision Tree', 'Random Forest', 'XGBoost']:
    shap_explainer = shap.TreeExplainer(best_model)
    shap_values_raw = shap_explainer.shap_values(X_test_scaled)
    shap_values_class1, base_value = extract_shap_class1(shap_values_raw, shap_explainer)
    X_shap_display = X_test
else:
    shap_explainer = shap.KernelExplainer(best_model.predict_proba, X_train_scaled.iloc[:100])
    shap_values_raw = shap_explainer.shap_values(X_test_scaled.iloc[:50])
    shap_values_class1, base_value = extract_shap_class1(shap_values_raw, shap_explainer)
    X_shap_display = X_test.iloc[:50]

assert shap_values_class1.shape[0] == len(X_shap_display), \
    f"Shape mismatch: SHAP {shap_values_class1.shape[0]} vs display {len(X_shap_display)}"

print(f"SHAP explainer created for: {best_model_name}")
print(f"SHAP values shape: {shap_values_class1.shape}, Display data shape: {X_shap_display.shape}")

# Global Feature Importance (bar plot)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class1, X_shap_display, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_model_name}')
plt.tight_layout()
plt.show()

# SHAP Beeswarm (dot) plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class1, X_shap_display, show=False)
plt.title(f'SHAP Beeswarm Plot - {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Local Explanation - Waterfall
idx = 0
print(f"\nSample {idx}: Actual={y_test.iloc[idx]}, Predicted={best_model.predict(X_test_scaled.iloc[[idx]])[0]}")

plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_class1[idx],
        base_values=base_value,
        data=X_shap_display.iloc[idx],
        feature_names=X.columns.tolist()
    ),
    max_display=13
)
plt.tight_layout()
plt.show()

joblib.dump(shap_explainer, 'shap_explainer.pkl')
print("SHAP explainer saved to: shap_explainer.pkl")

In [ ]:
# ============================================
# LIME EXPLAINABILITY - Local Explanation
# ============================================

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=X.columns.tolist(),
    class_names=['Rejected', 'Approved'],
    mode='classification',
    random_state=42
)

idx = 0
lime_exp = lime_explainer.explain_instance(
    data_row=X_test_scaled.iloc[idx].values,
    predict_fn=best_model.predict_proba,
    num_features=13
)

print(f"LIME Explanation for Sample {idx}:")
print(f"  Actual: {y_test.iloc[idx]}, Predicted: {best_model.predict(X_test_scaled.iloc[[idx]])[0]}")
print()

fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(12, 6)
plt.title(f'LIME Explanation - {best_model_name} - Sample {idx}')
plt.tight_layout()
plt.show()

print("\nFeature contributions:")
for feat, weight in lime_exp.as_list():
    direction = "-> Approved" if weight > 0 else "-> Rejected"
    print(f"  {feat}: {weight:+.4f} {direction}")

joblib.dump(lime_explainer, 'lime_explainer.pkl')
print("\nLIME explainer saved to: lime_explainer.pkl")

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import pandas as pd
import numpy as np
import shap

# ============================================
# USE IN-MEMORY OBJECTS FROM TRAINING
# ============================================
gui_model = best_model
gui_scaler = scaler
gui_model_name = best_model_name
gui_shap_explainer = shap_explainer
print(f"Using in-memory model: {gui_model_name}")

FEATURE_COLUMNS = [
    'no_of_dependents', 'education', 'self_employed', 'income_annum',
    'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value',
    'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value',
    'income_loan_ratio', 'total_assets'
]

VALIDATION_RULES = {
    'income':     {'min': 0,   'max': 100_000_000, 'name': 'Annual Income'},
    'loan':       {'min': 0,   'max': 100_000_000, 'name': 'Loan Amount'},
    'cibil':      {'min': 300, 'max': 900,          'name': 'CIBIL Score'},
    'res_assets': {'min': -50_000_000, 'max': 500_000_000, 'name': 'Residential Assets'},
    'com_assets': {'min': -50_000_000, 'max': 500_000_000, 'name': 'Commercial Assets'},
    'lux_assets': {'min': -50_000_000, 'max': 500_000_000, 'name': 'Luxury Assets'},
    'bank_assets':{'min': -50_000_000, 'max': 500_000_000, 'name': 'Bank Assets'},
}

def validate_range(value, key):
    rule = VALIDATION_RULES.get(key)
    if rule and (value < rule['min'] or value > rule['max']):
        return f"{rule['name']} must be between {rule['min']:,} and {rule['max']:,} (got {value:,.0f})"
    return None

def extract_gui_shap(sv):
    """Extract 1-D SHAP vector for the positive class from a single-sample prediction."""
    if isinstance(sv, list):
        return np.asarray(sv[1]).flatten()
    sv_arr = np.asarray(sv)
    if sv_arr.ndim == 3:
        return sv_arr[0, :, 1]
    return sv_arr.flatten()

def predict_loan():
    try:
        for entry in numeric_entries:
            entry.configure(style='TEntry')

        dependents = float(dep_spin.get())
        education = 1.0 if edu_combo.get() == "Graduate" else 0.0
        self_emp = 1.0 if self_combo.get() == "Yes" else 0.0
        income = float(income_entry.get())
        loan_amount = float(loan_entry.get())
        loan_term = float(term_spin.get())
        cibil = float(cibil_entry.get())
        res_assets = float(res_entry.get())
        com_assets = float(com_entry.get())
        lux_assets = float(lux_entry.get())
        bank_assets = float(bank_entry.get())

        validations = [
            (validate_range(income, 'income'), income_entry),
            (validate_range(loan_amount, 'loan'), loan_entry),
            (validate_range(cibil, 'cibil'), cibil_entry),
            (validate_range(res_assets, 'res_assets'), res_entry),
            (validate_range(com_assets, 'com_assets'), com_entry),
            (validate_range(lux_assets, 'lux_assets'), lux_entry),
            (validate_range(bank_assets, 'bank_assets'), bank_entry),
        ]

        errors = []
        for error_msg, entry_widget in validations:
            if error_msg:
                errors.append(error_msg)
                entry_widget.configure(style='Error.TEntry')

        if errors:
            messagebox.showwarning("Validation Error", "\n".join(errors))
            return

        income_loan_ratio = 0.0 if loan_amount == 0 else income / loan_amount
        total_assets = res_assets + com_assets + lux_assets + bank_assets

        input_data = pd.DataFrame([[
            dependents, education, self_emp, income, loan_amount,
            loan_term, cibil, res_assets, com_assets, lux_assets,
            bank_assets, income_loan_ratio, total_assets
        ]], columns=FEATURE_COLUMNS)

        input_scaled = gui_scaler.transform(input_data)
        input_scaled_df = pd.DataFrame(input_scaled, columns=FEATURE_COLUMNS)

        prediction = gui_model.predict(input_scaled)
        probability = gui_model.predict_proba(input_scaled)[0][1]

        sv = gui_shap_explainer.shap_values(input_scaled_df)
        sv_class1 = extract_gui_shap(sv)

        feature_contributions = sorted(
            zip(FEATURE_COLUMNS, sv_class1),
            key=lambda x: abs(float(x[1])), reverse=True
        )

        explanation_lines = []
        for feat_name, shap_val in feature_contributions[:6]:
            shap_val = float(shap_val)
            direction = "towards Approved" if shap_val > 0 else "towards Rejected"
            explanation_lines.append(f"  {feat_name}: {shap_val:+.4f} ({direction})")

        explanation = "\n".join(explanation_lines)

        if int(prediction[0]) == 1:
            result = "LOAN APPROVED"
            color = "green"
        else:
            result = "LOAN REJECTED"
            color = "red"

        prob_bar['value'] = probability * 100
        prob_label.config(text=f"{probability:.1%}")

        result_text.config(state='normal')
        result_text.delete('1.0', 'end')
        result_text.insert('end', f"{result}\n\n", 'result')
        result_text.insert('end', f"Approval Probability: {probability:.4f}\n\n")
        result_text.insert('end', f"Model Used: {gui_model_name}\n\n")
        result_text.insert('end', "SHAP Feature Contributions (Real XAI):\n\n")
        result_text.insert('end', explanation)
        result_text.tag_config('result', foreground=color, font=("Arial", 16, "bold"))
        result_text.config(state='disabled')

    except ValueError as e:
        messagebox.showerror("Input Error", f"Please enter valid numeric values.\n{e}")
    except Exception as e:
        messagebox.showerror("Error", str(e))

# ============================================
# GUI LAYOUT
# ============================================
root = tk.Tk()
root.title(f"Loan Approval Prediction ({gui_model_name})")
root.geometry("900x1020")
root.configure(bg="#f4f6f8")

style = ttk.Style()
style.configure('Error.TEntry', fieldbackground='#ffcccc')

heading = tk.Label(root, text="Loan Approval Prediction System",
                   font=("Arial", 22, "bold"), bg="#f4f6f8", fg="#1f2937")
heading.pack(pady=15)

subtitle = tk.Label(root, text=f"Model: {gui_model_name} | Powered by SHAP",
                     font=("Arial", 11), bg="#f4f6f8", fg="#6b7280")
subtitle.pack()

frame = tk.Frame(root, bg="white", bd=2, relief="ridge")
frame.pack(padx=20, pady=10, fill="both", expand=False)

numeric_entries = []

def create_label(text, row):
    label = tk.Label(frame, text=text, font=("Arial", 11), bg="white", anchor="w")
    label.grid(row=row, column=0, padx=10, pady=6, sticky="w")

def create_entry(row):
    entry = ttk.Entry(frame, font=("Arial", 11), width=25)
    entry.grid(row=row, column=1, padx=10, pady=6)
    numeric_entries.append(entry)
    return entry

create_label("Number of Dependents", 0)
dep_spin = ttk.Spinbox(frame, from_=0, to=5, width=23, font=("Arial", 11))
dep_spin.set(0)
dep_spin.grid(row=0, column=1, padx=10, pady=6)

create_label("Education", 1)
edu_combo = ttk.Combobox(frame, values=["Graduate", "Not Graduate"],
                          state="readonly", width=22, font=("Arial", 11))
edu_combo.set("Graduate")
edu_combo.grid(row=1, column=1, padx=10, pady=6)

create_label("Self Employed", 2)
self_combo = ttk.Combobox(frame, values=["Yes", "No"],
                           state="readonly", width=22, font=("Arial", 11))
self_combo.set("No")
self_combo.grid(row=2, column=1, padx=10, pady=6)

create_label("Annual Income", 3)
income_entry = create_entry(3)

create_label("Loan Amount", 4)
loan_entry = create_entry(4)

create_label("Loan Term (years)", 5)
term_spin = ttk.Spinbox(frame, from_=2, to=30, width=23, font=("Arial", 11))
term_spin.set(12)
term_spin.grid(row=5, column=1, padx=10, pady=6)

create_label("CIBIL Score (300-900)", 6)
cibil_entry = create_entry(6)

create_label("Residential Assets Value", 7)
res_entry = create_entry(7)

create_label("Commercial Assets Value", 8)
com_entry = create_entry(8)

create_label("Luxury Assets Value", 9)
lux_entry = create_entry(9)

create_label("Bank Asset Value", 10)
bank_entry = create_entry(10)

predict_btn = tk.Button(root, text="Predict Loan Status",
                        font=("Arial", 14, "bold"), bg="#2563eb", fg="white",
                        padx=20, pady=10, command=predict_loan, cursor="hand2")
predict_btn.pack(pady=10)

gauge_frame = tk.Frame(root, bg="#f4f6f8")
gauge_frame.pack(pady=(0, 5))
tk.Label(gauge_frame, text="Approval Probability:", font=("Arial", 11),
         bg="#f4f6f8").pack(side='left', padx=(0, 10))
prob_bar = ttk.Progressbar(gauge_frame, length=300, mode='determinate',
                            maximum=100)
prob_bar.pack(side='left')
prob_label = tk.Label(gauge_frame, text="--", font=("Arial", 11, "bold"),
                       bg="#f4f6f8")
prob_label.pack(side='left', padx=(10, 0))

result_text = scrolledtext.ScrolledText(root, font=("Consolas", 11),
                                         width=80, height=12, relief="solid",
                                         bd=2, state='disabled', wrap='word')
result_text.pack(padx=20, pady=10)

root.mainloop()

# Summary & Conclusions

## Project Overview
This notebook implements a **Loan Approval Prediction System** using 5 machine learning models trained on applicant demographic and financial data.

## Pipeline
1. **Data Loading** — Dynamic path resolution, no hardcoded paths
2. **EDA** — Class balance, negative values, IQR outliers, correlation heatmap, feature distributions
3. **Preprocessing** — Column cleanup, feature engineering (`income_loan_ratio`, `total_assets`), explicit label encoding with assertions
4. **Scaling** — `StandardScaler` applied uniformly to all models (fit on train only)
5. **Modeling** — 5 models with `GridSearchCV` hyperparameter tuning:
   - Decision Tree, Logistic Regression, Random Forest, XGBoost, KNN
6. **Evaluation** — 7 metrics (Accuracy, Precision, Recall, F1, AUC-ROC, CV mean, CV std), confusion matrices, ROC curves, feature importance
7. **Model Selection** — Best model chosen by cross-validation accuracy (not test accuracy)
8. **Explainability** — SHAP (global bar + beeswarm + local waterfall) and LIME (local feature contributions)
9. **Deployment** — Tkinter GUI with real SHAP-powered explanations per prediction

## Key Findings
- **Best Model**: Dynamically selected based on CV accuracy
- **Most Important Features**: Identified via SHAP — check the beeswarm plot above for the actual ranking on your dataset run
- **Class Balance**: Dataset distribution analyzed and stratified splitting ensures representative train/test sets
- **Data Quality**: Negative asset values detected and reported; outliers quantified via IQR

## Reproducibility
- `random_state=42` used across all splits, models, and cross-validation
- All models saved to `.pkl` files for deployment
- SHAP and LIME explainers persisted for GUI usage

## Files Produced
| File | Contents |
|---|---|
| `loan_model.pkl` | Best-performing model (selected by CV accuracy) |
| `scaler.pkl` | Fitted StandardScaler |
| `best_model_name.pkl` | Name string of the selected model |
| `shap_explainer.pkl` | SHAP explainer for the best model |
| `lime_explainer.pkl` | LIME explainer for local explanations |